# 02 · Train classical targets

**Purpose.** Train the five classical target recipes on the 10,000 member images and evaluate them on the 10,000 non-members.

**Inputs.** Feature cache, split file, `config/experiment.yaml`.

**Outputs.** `models/classical/`, `models/probs/`, `models` rows for C1 (1 run), C2 (3 seeds × 8 checkpoints), C3 and C4 (3 seeds each), C5 (1 run).

**Syllabus topics.** Euclidean/affine transforms (C4 augmentation); PCA; discriminant-function classification (softmax, Fisher LDA); nearest-neighbour classification (C1, a CBIR-style retrieval engine).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
from mia.training import Workspace, run_jobs

ws = Workspace.open()
exp = ws.exp
jobs = [(cid, seed, None) for cid in ("C1", "C2", "C3", "C4", "C5") for seed in exp.seeds_for(exp.configs[cid])]
summary = run_jobs(ws, jobs)

## Target table

In [ ]:
from mia import db

m = db.models_df(ws.conn, role="target")
m = m[m["config_id"].isin(["C1", "C2", "C3", "C4", "C5"])]
table = m[["model_id", "config_id", "seed", "epoch", "train_acc", "test_acc", "gap_acc", "loss_ratio", "n_params",
           "infer_ms_per_img", "train_seconds"]].round(4)
table

In [ ]:
print(f"Trained {summary['trained']}, skipped {summary['skipped']}, failed {summary['failed']} "
      f"(errors, if any, in results/train_errors.log)")
final = table[table["epoch"].isna() | (table["epoch"] == 200)]
print(final.groupby("config_id")[["train_acc", "test_acc", "gap_acc"]].mean().round(3))

## Results
The summary above gives mean train and test accuracy per configuration: C1 memorises (train accuracy 100%), C2 overfits, and PCA, L2, augmentation and LDA shrink the gap step by step.